# Emotion Contagion ABM

This notebook is a light front end for the refactored simulation project.

Use it when you want a notebook workflow, but keep the core engine in the Python modules.

In [11]:
from pathlib import Path
import sys
import pickle

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = CURRENT_DIR.parent
RUNNING_DIR = PROJECT_ROOT / "running"
SRC_DIR = PROJECT_ROOT / "src"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"

for path in [str(PROJECT_ROOT), str(RUNNING_DIR), str(SRC_DIR)]:
    if path not in sys.path:
        sys.path.insert(0, path)

from metrics import plot_sentiment_evolution, summary_dataframe_from_batch

## 1. Load a YAML config and run one condition or a small batch

In [12]:
def load_result_pickle(path: str | Path):
    path = Path(path)
    with path.open("rb") as f:
        return pickle.load(f)


def load_all_result_pickles(results_dir: str | Path):
    results_dir = Path(results_dir)
    pickle_paths = sorted(results_dir.rglob("simulation_result_run_*.pkl"))
    if not pickle_paths:
        raise FileNotFoundError(f"No result pickle files were found under: {results_dir}")
    return [load_result_pickle(path) for path in pickle_paths]

In [21]:
results_dir = r"C:\Users\sarah\Downloads\Emotion-Contagion\BigProject\outputs"   # change this to your actual batch output folder
results_list = load_all_result_pickles(results_dir)

print("Loaded result files:", len(results_list))

Loaded result files: 180


## 2. Quick summary of the runs you just saved

In [22]:
summary_rows = []

for results in results_list:
    final_avg = results.avg_emotion_history[-1] if results.avg_emotion_history else None
    final_member_emotions = [agent["emotion"] for agent in results.state.agents if agent.get("role") == "member"]
    leader = results.state.agents[results.state.leader_index]

    summary_rows.append(
        {
            "run_id": results.run_id,
            "seed": results.seed,
            "condition_name": results.metadata.get("condition_name"),
            "structure": results.metadata.get("structure"),
            "leader_style": results.metadata.get("leader_style"),
            "max_iterations": results.metadata.get("max_iterations"),
            "num_interventions": len(results.intervention_timesteps),
            "total_interactions": sum(results.interactions_per_timestep),
            "final_avg_member_emotion": final_avg,
            "final_min_member_emotion": min(final_member_emotions) if final_member_emotions else None,
            "final_max_member_emotion": max(final_member_emotions) if final_member_emotions else None,
            "leader_emotion": leader.get("emotion"),
            "leader_threshold": leader.get("interventionThreshold"),
        }
    )

summary_df = pd.DataFrame(summary_rows)
summary_df.head()

,run_id,seed,condition_name,structure,leader_style,max_iterations,num_interventions,total_interactions,final_avg_member_emotion,final_min_member_emotion,final_max_member_emotion,leader_emotion,leader_threshold
0,100,19,structure_community__leader_style_High_Initial...,community,High_Initially_Constrained,50,0,287,-0.075521,-0.196484,-0.040283,1.0,-0.5
1,81,0,structure_community__leader_style_High_Initial...,community,High_Initially_Constrained,50,16,283,-0.593250,-0.997609,-0.156238,1.0,-0.5
2,82,1,structure_community__leader_style_High_Initial...,community,High_Initially_Constrained,50,0,300,-0.331630,-0.680366,-0.110941,1.0,-0.5
3,83,2,structure_community__leader_style_High_Initial...,community,High_Initially_Constrained,50,21,282,-0.686324,-0.998166,-0.216232,1.0,-0.5
4,84,3,structure_community__leader_style_High_Initial...,community,High_Initially_Constrained,50,0,315,-0.408920,-1.000000,-0.087319,1.0,-0.5


In [23]:
summary_df.shape

(180, 13)